# Archival thesis notebook
Outputs and private paths removed. See this folder's README for known limitations. The original dataset is not included.


In [ ]:
!pip install keras

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer
from keras.layers import SpatialDropout1D
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import GridSearchCV
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout
from keras.layers import Bidirectional
from keras.layers import GlobalMaxPool1D
from keras.layers import Embedding
from keras.preprocessing import sequence
from keras.callbacks import EarlyStopping
from sklearn.feature_selection import RFE
import re

In [ ]:
df = pd.read_csv('../../data/private/author_paragraphs.csv')

In [ ]:
df = df.dropna(how='any',axis=0)
df.drop_duplicates(inplace = True)
df.head()

In [ ]:
df['Author'].value_counts( )

In [ ]:
df.dtypes

In [ ]:
possible_labels = df.Author.unique()
label_dict = {}
for index, possible_label in enumerate(possible_labels):
    label_dict[possible_label] = index
label_dict

In [ ]:
df['Label'] = df.Author.replace(label_dict)

In [ ]:
df.head( )

In [ ]:
df.Label

In [ ]:
df.describe()

In [ ]:
# Checking NA counts
df.isnull().sum()

In [ ]:
# Visualize the depth of dataset and showing the value of each class contains.

# Look at classification type counts
counts = df.Label.value_counts()
print(counts)

#plot counts
plt.figure()
sns.distplot(counts, hist = False, color = 'purple')
plt.ylabel('% of records')
plt.show()

In [ ]:
# Find the highest length of a news.
max_length=df["Paragraph"].str.len().max()
print(max_length)

In [ ]:
# Find the labels that the dataset contains.
train=df
a=train.Label.unique()
l=len(a)
print(a)

In [ ]:
# Convert the labels into Integers.
for i in range (l):
    train.Label = train.Label.replace(a[i], i)

In [ ]:
# Showing portion of the dataset from the begining.

train.Paragraph.head()

In [ ]:
# showing the value of each class contains
train.Paragraph.value_counts()

In [ ]:
# Tokenizing Process

MAX_NB_WORDS = 50000
MAX_SEQUENCE_LENGTH = 600
EMBEDDING_DIM = 100
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~।', lower=False)
tokenizer.fit_on_texts(df.Paragraph.values)
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

In [ ]:
X = tokenizer.texts_to_sequences(df.Paragraph.values)
X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)
print('Shape of data tensor:', X.shape)

In [ ]:
Y = pd.get_dummies(df.Label).values
print('Shape of label tensor:', Y.shape)

In [ ]:
# Splitting the dataset for training(70%) and testing(30%).

train_features, test_features, train_labels, test_labels = train_test_split(X, Y, test_size=.30)

In [ ]:
# The model of LSTM

model = Sequential()
model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
model.add(LSTM(64))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(l, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

In [ ]:
# Setting the Epoch and Batch Size
# Epoch: The number of epochs is a hyperparameter that defines the number times that the learning algorithm will work through the entire training dataset.
# Batch Size: The batch size is a hyperparameter that defines the number of samples to work through before updating the internal model parameters.

epochs = 10
batch_size = 128

In [ ]:
# Training model

history = model.fit(train_features, train_labels, epochs=epochs, batch_size=batch_size,validation_split=0.1,callbacks=[EarlyStopping(monitor='val_loss', patience=3, min_delta=0.0001)])

In [ ]:
# Evaluation for Accuracy

accr = model.evaluate(test_features,test_labels)
print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0],accr[1]))

In [ ]:
# Plotting Training vs Validation Graph for Loss and Accuracy

from matplotlib import pyplot as plt
plt.title('Loss')
plt.plot(history.history['loss'],'r', label='train')
plt.plot(history.history['val_loss'],'b', label='test')
plt.legend()
plt.show()
plt.title('Accuracy')
plt.plot(history.history['accuracy'],'r', label='train')
plt.plot(history.history['val_accuracy'],'b', label='test')
plt.legend()
plt.show()

In [ ]:
# Evaluation for Accuracy for calculating precession, recall and f1-score.

predicted_classes = model.predict(test_features)
predicted_classes = np.argmax(np.round(predicted_classes),axis=1)
#test_Y = np.argmax(np.round(test_labels),axis=1)

In [ ]:
predicted_classes.shape, test_labels.shape
print(test_labels.shape)

In [ ]:
test_Y=[]
for i in range (1038):
    for j in range (6):
        if(test_labels[i][j] == 1):
            test_Y.append(j)
print(len(test_Y))

In [ ]:
from sklearn.metrics import classification_report
target_names = ["Class {}".format(i) for i in range(6)]
print(classification_report(test_Y, predicted_classes, target_names=target_names))

In [ ]:
def pplotConfusionMatrix(cm, accuracy, name):
    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r');
    plt.ylabel('Actual label');
    plt.xlabel('Predicted label');
    all_sample_title = '{0}\nAccuracy Score: {1}'.format(name, accuracy[1])
    plt.title(all_sample_title, size = 15)

In [ ]:
cm = confusion_matrix(test_Y, predicted_classes)
pplotConfusionMatrix(cm = cm, accuracy = accr,name='LSTM')

In [ ]:
from operator import truediv
tp = np.diag(cm)
prec = list(map(truediv, tp, np.sum(cm, axis=0)))
rec = list(map(truediv, tp, np.sum(cm, axis=1)))
print("Accuracy = ",tp.sum( )/cm.sum( ))
print("Precision = {}\nRecall = {}".format(prec, rec))
print("Avg Precision = {}\nAvg Recall = {}".format(np.sum(prec)/2, np.sum(rec)/2))